# Settings

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
import scipy as sc
import scipy.stats as stats
import astropy.cosmology as ac
from matplotlib.patches import Ellipse
from matplotlib.gridspec import GridSpec
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numdifftools as nd
import pandas as pd
import sys
import astropy.constants as cst
import random
from matplotlib.patches import Patch
import zeus
from chainconsumer import ChainConsumer
from tqdm import tqdm
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from astropy.io import fits
from tqdm.notebook import tqdm
from IPython.display import display, Markdown, Math

import astropy.units as u
from scipy.integrate import quad

In [2]:
import jax
import jax.numpy as jnp
import jax_cosmo as jc

/home/cl-a-millard/venvs/jupyter/lib/python3.14/site-packages/jax_cosmo/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound


In [ ]:
import fisher_matrix_analysis as fma
import fisher_matrix_analysis_jax as fma_jax
import jax_optimizer as rw_fma_jax

# Data

In [4]:
# define cosmological parameters
H0_fid=73
Mb_fid=-19.3
c=float(np.array(cst.c.to('km/s')))


#for flat LCDM, according to Planck 2018 (TT,TE,EE+lowE+lensing)
Om0_fLCDM = 0.3153
Ode0_fLCDM = 1-Om0_fLCDM

#for LCDM
Om0_kLCDM, Ode0_kLCDM = 0.3153, 0.6
Ok0_kLCDM = 1-Om0_kLCDM-Ode0_kLCDM

#for flat wCDM
Om0_fwCDM, w0_fwCDM = 0.3153, -1.029

#for k-wCDM
Om0_kwCDM, Ode0_kwCDM, w0_kwCDM =  0.3153, 0.6, -1.029

#for flat CPL
Om0_fCPL, w0_fCPL, wa_fCPL = 0.3153, -1.029, 0.21

#for CPL
Om0_kCPL, Ode0_kCPL, w0_kCPL, wa_kCPL = 0.3153, 0.6, -1.029, 0.21

#for PEDE
Om0_PEDE, Ode0_PEDE = 0.3153, 0.6
Ok0_PEDE=1-Om0_PEDE-Ode0_PEDE

fiducial_fLCDM  = [Om0_fLCDM,Mb_fid]

fiducial_kLCDM = [Om0_kLCDM, Ode0_kLCDM, Mb_fid]


fiducial_fwCDM = [Om0_fwCDM, w0_fwCDM, Mb_fid]

fiducial_kwCDM = [Om0_kwCDM, Ode0_kwCDM, w0_kwCDM,Mb_fid]

fiducial_fCPL=[Om0_fCPL,w0_fCPL,wa_fCPL,Mb_fid]
 
# fiducial cosmologie
#LCDM
cosmo_fLCDM = ac.FlatLambdaCDM(H0=H0_fid,Om0=Om0_fLCDM)
cosmo_kLCDM = ac.LambdaCDM(H0=H0_fid,Om0=Om0_fLCDM,Ode0=Ode0_fLCDM)

#wCDM
cosmo_fwCDM = ac.FlatwCDM(H0=H0_fid,Om0=Om0_fLCDM,w0=w0_fwCDM)
cosmo_kwCDM = ac.wCDM(H0=H0_fid,Om0=Om0_kwCDM,Ode0=Ode0_kwCDM,w0=w0_kwCDM)

#CPL
cosmo_fCPL=ac.Flatw0waCDM(H0=H0_fid,Om0=Om0_fCPL,w0=w0_fCPL,wa=wa_fCPL)

In [5]:
# Roman data

roman = pd.read_csv('roman_data/mock_roman.csv', sep=';')
z_roman = roman.z.values
dist_roman = roman.Nstar.values
#dist_roman = np.full_like(np.array(z_roman),200) # fake flat distribution
cov = fits.getdata("roman_data/roman_cov.fits")

# last row is the redshifts -> no it is the first one actually
Cov_roman = cov[1:, :]
Cov_roman = np.asarray(Cov_roman).astype(np.float64)
II = np.eye(Cov_roman.shape[0])
Cov_inv_roman = np.linalg.solve(Cov_roman, II)
sigma_roman = np.sqrt(np.diag(Cov_roman))

# define bin edges
edges_mid = 0.5 * (z_roman[1:] + z_roman[:-1])
# first and last edges
first_edge = z_roman[0] - (edges_mid[0] - z_roman[0])
last_edge  = z_roman[-1] + (z_roman[-1] - edges_mid[-1])

# combine
z_edges = np.concatenate([[first_edge], edges_mid, [last_edge]])

# Mock noisy data
mock_roman = np.load('roman_data/NGR_fCPL.npz')
mb_roman = mock_roman['mb']

In [6]:
cosmo = cosmo_fCPL
cosmo_name = 'fCPL'
fiducial_cosmo = fiducial_fCPL

# 1. chi-square function

In [7]:
chi2_fid_fma = fma.chi2(fiducial_cosmo,
                        z_roman,
                        mb_roman,
                        Cov_roman,
                        cosmo_name,
                        H0_fid)

chi2_fid_fma_jw = fma_jax.chi2(fiducial_cosmo,
                               z_roman,
                               mb_roman,
                               Cov_roman,
                               H0_fid)

chi2_fid_rw_fma_jw = rw_fma_jax.chi2(fiducial_cosmo, 
                                     z_roman, 
                                     mb_roman, 
                                     Cov_roman, 
                                     cosmo_name, 
                                     H0_fid)

print("=" * 60)
print("Fiducial χ² comparison")
print("=" * 60)

print(f"fma (original)       : {chi2_fid_fma:.10f}")
print(f"fma_jax              : {chi2_fid_fma_jw:.10f}")
print(f"rw_fma_jax           : {chi2_fid_rw_fma_jw:.10f}")

print("=" * 60)

Fiducial χ² comparison
fma (original)       : 0.0000000000
fma_jax              : 0.0000000795
rw_fma_jax           : 0.0000000795


# 2. Fisher Matrix

In [12]:
FF_fma = fma.fisher_matrix_observable(fiducial_cosmo, 
                                      Cov_inv_roman, 
                                      z_roman, 
                                      cosmo_name, 
                                      H0_fid)

FF_fma_jx = fma_jax.fisher_matrix_observable(fiducial_cosmo,
                                             Cov_inv_roman,
                                             z_roman,
                                             H0_fid)

FF_rw_fma_jx = rw_fma_jax.fisher_matrix(rw_fma_jax.chi2, 
                                        fiducial_cosmo,
                                        z_roman,
                                        mb_roman,
                                        Cov_roman,
                                        cosmo_name,
                                        H0_fid)

print("=" * 90)
print("Fisher matrix comparison")
print("=" * 90)

print(f"fma (original)       : {FF_fma}")
print(f"fma_jax              : {FF_fma_jx}")
print(f"rw_fma_jax           : {np.array(FF_rw_fma_jx)}")

print("=" * 90)

Fisher matrix comparison
fma (original)       : [[ 30550.74537293   9947.31885081   1892.13024034 -21953.37065059]
 [  9947.31885081   3689.34791854    648.86610168  -6999.30189908]
 [  1892.13024034    648.86610168    126.44977437  -1164.71617313]
 [-21953.37065059  -6999.30189908  -1164.71617313  26616.2800282 ]]
fma_jax              : [[ 30550.74916976   9947.31664392   1892.1297312  -21953.40140001]
 [  9947.31664392   3689.34460623    648.86567624  -6999.31660905]
 [  1892.1297312     648.86567624    126.44968897  -1164.71748829]
 [-21953.40140001  -6999.31660905  -1164.71748829  26616.28002833]]
rw_fma_jax           : [[ 30550.76556059   9947.31149835   1892.12981451 -21953.40140001]
 [  9947.31149835   3689.34353547    648.86463362  -6999.31660905]
 [  1892.12981451    648.86463362    126.44921958  -1164.71748829]
 [-21953.40140001  -6999.31660905  -1164.71748829  26616.28002833]]


In [13]:
ac.Planck13.H0

<Quantity 67.77 km / (Mpc s)>